In [306]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

plt.style.use('dark_background')
import warnings
warnings.filterwarnings('ignore')
%matplotlib inline

In [307]:
df = pd.read_csv('./content/data/train.csv')

In [308]:
df.columns

Index(['Id', 'MSSubClass', 'MSZoning', 'LotFrontage', 'LotArea', 'Street',
       'Alley', 'LotShape', 'LandContour', 'Utilities', 'LotConfig',
       'LandSlope', 'Neighborhood', 'Condition1', 'Condition2', 'BldgType',
       'HouseStyle', 'OverallQual', 'OverallCond', 'YearBuilt', 'YearRemodAdd',
       'RoofStyle', 'RoofMatl', 'Exterior1st', 'Exterior2nd', 'MasVnrType',
       'MasVnrArea', 'ExterQual', 'ExterCond', 'Foundation', 'BsmtQual',
       'BsmtCond', 'BsmtExposure', 'BsmtFinType1', 'BsmtFinSF1',
       'BsmtFinType2', 'BsmtFinSF2', 'BsmtUnfSF', 'TotalBsmtSF', 'Heating',
       'HeatingQC', 'CentralAir', 'Electrical', '1stFlrSF', '2ndFlrSF',
       'LowQualFinSF', 'GrLivArea', 'BsmtFullBath', 'BsmtHalfBath', 'FullBath',
       'HalfBath', 'BedroomAbvGr', 'KitchenAbvGr', 'KitchenQual',
       'TotRmsAbvGrd', 'Functional', 'Fireplaces', 'FireplaceQu', 'GarageType',
       'GarageYrBlt', 'GarageFinish', 'GarageCars', 'GarageArea', 'GarageQual',
       'GarageCond', 'PavedDrive

In [309]:
print(f'Count features with null-value = {df.isnull().any(axis=0).sum()}')

Count features with null-value = 19


Очень много фичей, много интересных мыслей появлялось в процессе анализа. Щас всё попробую. Но сначала надо избавиться от null-значений.

# Работа с null-значений
В начале анализа выяснил, что много фичей имеют много null-значений. Сначала казалось это плохо, но потом узнал, что эти null-значения показывают отсутствие данного
объекта на участке (напр, нет забора, камина и т.п.). Это не значит что данные были потеряны.\
Поэтому важно все эти null-значения заменить на что-то логично, с чем модели могут работать.

In [310]:
null_count = df.isnull().sum()
null_count[null_count > 0]

LotFrontage      259
Alley           1369
MasVnrType       872
MasVnrArea         8
BsmtQual          37
BsmtCond          37
BsmtExposure      38
BsmtFinType1      37
BsmtFinType2      38
Electrical         1
FireplaceQu      690
GarageType        81
GarageYrBlt       81
GarageFinish      81
GarageQual        81
GarageCond        81
PoolQC          1453
Fence           1179
MiscFeature     1406
dtype: int64

## LotFrontage null
Показывает ширину участка, граничащую с улицей.\
Очевидно, дом не может не граничить с улицей, т.к. какой-то подход должен к нему быть. Поэтому стоит эти значения заполнить.

Можно было бы заполнить все null-значения обычным средним, но хочется большей точности. У каждого сэмпла есть *LotConfig*, которые показывает, насколько
хорошо дом прилегает к дороге. Думаю, у домов, которые прилегают к дороге с 3-х сторон, значение *LotFrontage* будет заметно выше:

In [311]:
df.groupby('LotConfig')['LotFrontage'].mean()

LotConfig
Corner     84.039801
CulDSac    59.911111
FR2        63.515152
FR3        70.750000
Inside     67.715686
Name: LotFrontage, dtype: float64

Интересно, что угловой участок (*Corner*) имеет большее значение, чем участок, окружённый улицами с 3-х сторон (*FR3*). Но оба они всё равно имеют большее значение,
чем другие.

In [312]:
lotfrontage_null = df.groupby('LotConfig')['LotFrontage'].mean().round()

for value, mean in lotfrontage_null.items():
    df.loc[(df['LotFrontage'].isnull()) & (df['LotConfig'] == value), 'LotFrontage'] = mean

## Alley null
Показывает либо материал аллеи, либо её отсутствие. Здесь null-значение несёт важную информацию.

In [313]:
df.loc[df['Alley'].isnull(), 'Alley'] = 'Absent'

## MasVnrType null
Показывает облицовку дома. Null-значение несёт в себе информацию.

In [314]:
df.loc[df['MasVnrType'].isnull(), 'MasVnrType'] = 'Absent'

## MasVnrArea null
Показывает площадь облицовки из кирпица. Имеет 8 null-значений, хотя отсутствие облицовки должно просто обозначатьс цифрой 0.

In [315]:
df[df['MasVnrArea'].isnull()][['MasVnrType', 'MasVnrArea']]

,MasVnrType,MasVnrArea
234,Absent,NaN
529,Absent,NaN
650,Absent,NaN
936,Absent,NaN
973,Absent,NaN
977,Absent,NaN
1243,Absent,NaN
1278,Absent,NaN


У null-значений облицовка отсутствует, так что можно спокойно присваивать значение 0:

In [316]:
df.loc[df['MasVnrArea'].isnull(), 'MasVnrArea'] = 0

## BsmtQual null
Показывает высоту подвала. Null-значения показывают отсутствие подвала.

In [317]:
df.loc[df['BsmtQual'].isnull(), 'BsmtQual'] = 'Absent'

## BsmtCond null
Оценивает общее состояние подвала. Null-значение показывает отсутствие подвала.

In [318]:
df.loc[df['BsmtCond'].isnull(), 'BsmtCond'] = 'Absent'

## BsmtExposure null
Показывает стену, ведущую на террасу или в сад. Т.е. насколько хорошо подвал выходит на наружу: его может быть совсем не видно, могут быть небольшие окна, из которых попадает свет,
а может быть целая дверь с выходом во двор.

У 37 домов подвалы отсутсвуют, точно будет 37 null-значений. Но есть и ещё один:

In [319]:
df[(df['BsmtExposure'].isnull()) & (df['BsmtQual'] != 'Absent')].filter(regex=r'.*Bsmt.*')

,BsmtQual,BsmtCond,BsmtExposure,BsmtFinType1,BsmtFinSF1,BsmtFinType2,BsmtFinSF2,BsmtUnfSF,TotalBsmtSF,BsmtFullBath,BsmtHalfBath
948,Gd,TA,NaN,Unf,0,Unf,0,936,936,0,0


Видно, что подвал есть и достаточно большой. Но значения *BsmtFin\** показывают, что подвал недостроен. Поэтому фактически он есть, но невозможно использовать.\
Его тоже можно пометить как *Absent*:

In [320]:
df.loc[df['BsmtExposure'].isnull(), 'BsmtExposure'] = 'Absent'

## BsmtFinType1 null
Качество отделки основной зоны подвала. 37 null-значений. Те самые дома, у которых подвал отсутствует.

In [321]:
df.loc[df['BsmtFinType1'].isnull(), 'BsmtFinType1'] = 'Absent'

## BsmtFinType2 null
Качество отделки дополнительной зоны подвала. 38 null-значений. У 37 домов точно нет подвала. 38-й сэмпл очень интересный:

In [322]:
df[(df['BsmtFinType2'].isnull()) & (df['BsmtQual'] != 'Absent')].filter(regex=r'.*Bsmt.*')

,BsmtQual,BsmtCond,BsmtExposure,BsmtFinType1,BsmtFinSF1,BsmtFinType2,BsmtFinSF2,BsmtUnfSF,TotalBsmtSF,BsmtFullBath,BsmtHalfBath
332,Gd,TA,No,GLQ,1124,NaN,479,1603,3206,1,0


Это какое-то потерянное значение. У данного дома есть подвал, от отделан, достаточно хорошая площадь. Даже видно, что дополнительная зона подвала готова
к использованию (судя по *BsmtFinSF2*), но при этом значение *BsmtFinType2* отсутствует. Заполним его самым популярным значением в *BsmtFinType2*:

In [323]:
df['BsmtFinType2'].value_counts()

BsmtFinType2
Unf    1256
Rec      54
LwQ      46
BLQ      33
ALQ      19
GLQ      14
Name: count, dtype: int64

Самое популярное значение *Unf*, но мы видим, что дополнительная зона подвала точно готова к использованию. Использую лучше значение *Rec*:

In [324]:
df.loc[(df['BsmtFinType2'].isnull()) & (df['BsmtFinSF2'] > 0), 'BsmtFinType2'] = 'Rec'

In [325]:
df.loc[df['BsmtFinType2'].isnull(), 'BsmtFinType2'] = 'Absent'

## Electrical null
Показывает тип электрической системы.

In [326]:
df[df['Electrical'].isnull()]

,Id,MSSubClass,MSZoning,LotFrontage,LotArea,Street,Alley,LotShape,LandContour,Utilities,...,PoolArea,PoolQC,Fence,MiscFeature,MiscVal,MoSold,YrSold,SaleType,SaleCondition,SalePrice
1379,1380,80,RL,73.0,9735,Pave,Absent,Reg,Lvl,AllPub,...,0,NaN,NaN,NaN,0,5,2008,WD,Normal,167500


Ничего необычного, заполню самым частым значением:

In [327]:
df['Electrical'].value_counts()

Electrical
SBrkr    1334
FuseA      94
FuseF      27
FuseP       3
Mix         1
Name: count, dtype: int64

In [328]:
df.loc[df['Electrical'].isnull(), 'Electrical'] = 'SBrkr'

## FireplaceQu null
Показывает качество каминов. Null-значение показывает отсутствие камина в доме.

Проверим сэмплы, у которых качество камина имеет null-значение, а количество каминов > 0:

In [329]:
df[(df['FireplaceQu'].isnull()) & (df['Fireplaces'] > 0)]

,Id,MSSubClass,MSZoning,LotFrontage,LotArea,Street,Alley,LotShape,LandContour,Utilities,...,PoolArea,PoolQC,Fence,MiscFeature,MiscVal,MoSold,YrSold,SaleType,SaleCondition,SalePrice


В датасете отсутствуют подобные сэмплы, можно спокойно обозначать *Absent*:

In [330]:
df.loc[df['FireplaceQu'].isnull(), 'FireplaceQu'] = 'Absent'

## Garage* null
За гараж отвечают признаки *GarageType*, *GarageYrBlt*, *GarageFinist*, *GarageQual* и *GarageCond*.\
Каждый из них имеет 81 null-значений. Они показывают отсутствие гаража. Совпадение null-значений в каждом признаки показывает отсутствие пропусков.

In [331]:
df.loc[df['GarageType'].isnull() , 'GarageType'] = 'Absent'
df.loc[df['GarageFinish'].isnull() , 'GarageFinish'] = 'Absent'
df.loc[df['GarageQual'].isnull() , 'GarageQual'] = 'Absent'
df.loc[df['GarageCond'].isnull() , 'GarageCond'] = 'Absent'

И с фичой *GarageYrBlt* получается неоднозначная ситуация: столбец содержит числа (float64), однако при отсутствии гаража стоит null-значение. Изменить на *Absent* не получится.\
Если подобрать какое-то определённое число для null-значений (напр, 0 или год постройки дома), разные модели могут неправильно обучаться на них.

Думаю, хорошим вариантом будет изменить *GarageYrBlt* на возраст гаража *GarageAge*, а для наличия гаража использовать новую фичу *HasGarage*:

In [332]:
df['GarageAge'] = (df['YrSold'] - df['GarageYrBlt'])
df['GarageAge'] = df['GarageAge'].fillna(0)
df['GarageAge'] = df['GarageAge'].astype(int)

df['HasGarage'] = (df['GarageArea'] > 0).astype(int)

df = df.drop(columns='GarageYrBlt')

## PoolQC null
Показывает качество бассейна. Null-значение показывает отсутствие бассейна.

In [333]:
df.loc[df['PoolQC'].isnull(), 'PoolQC'] = 'Absent'

## Fence null
Показывает качество забора. Null-значение показывает отсутствие забора.

In [334]:
df.loc[df['Fence'].isnull(), 'Fence'] = 'Absent'

## MiscFeature null
Показывает наличие дополнительного функционала, которые не вошёл в основные фичи. Есть ещё фича *MiscValue*, которая показывает стоимость этого функционала.

Думаю, нет смысла указывать что за объект стоит на участке, если его ценность уже указана в *MiscValue*. А объкты, у которых нет ничего дополнительного, имеют
*MiscValue* = 0, что и показывает отсутствие каких-либо доп.объектов на участке.\
Поэтому удалю столбец *MiscFeature*:

In [335]:
df = df.drop(columns='MiscFeature')

# Преобразование фичей
_UPD_: Я тут щас преобразовывал фичи из категориальных в ординальные и понял, что не всем моделям подойдёт ординальное кодирование. Линейным моделям (а следовательно, и нейросетям)
лучше делать _one-hot-encoding_. Благо для этого есть _.get_dummies()_.\
Поэтому я продолжу здесь делать заготовки для _ordinal-encoding_, который пойдёт потом в пайплайн, но итоговое преобразование будет зависить от модели.

 ## MSSubClass fe
Признак содержит числа, обозначающие какой-то тип дома. Расположены в следующем порядке:
- 20  — 1-STORY 1946 & NEWER ALL STYLES
- 30  — 1-STORY 1945 & OLDER
- 40  — 1-STORY W/FINISHED ATTIC ALL AGES
- 45  — 1-1/2 STORY - UNFINISHED ALL AGES
- 50  — 1-1/2 STORY FINISHED ALL AGES
- 60  — 2-STORY 1946 & NEWER
- 70  — 2-STORY 1945 & OLDER
- 75  — 2-1/2 STORY ALL AGES
- 80  — SPLIT OR MULTI-LEVEL
- 85  — SPLIT FOYER
- 90  — DUPLEX - ALL STYLES AND AGES
- 120 — 1-STORY PUD (Planned Unit Development) - 1946 & NEWER
- 150 — 1-1/2 STORY PUD - ALL AGES
- 160 — 2-STORY PUD - 1946 & NEWER
- 180 — PUD - MULTILEVEL - INCL SPLIT LEV/FOYER
- 190 — 2 FAMILY CONVERSION - ALL STYLES AND AGES

Но возрастание числа не соответствует росту ценности дома.\
Можно было бы щтательно исследовать данную тему в интернете и поставить дома в возрастании в соответствии с данными. Но будет проще найти среднюю стоимость каждого дома и
расположить их в порядке возрастания средней стоимости дома:

In [336]:
df.groupby('MSSubClass', as_index=False)['SalePrice'].mean().sort_values('SalePrice')

,MSSubClass,SalePrice
1,30,95829.724638
13,180,102300.000000
3,45,108591.666667
14,190,129613.333333
10,90,133541.076923
12,160,138647.380952
4,50,143302.972222
9,85,147810.000000
2,40,156125.000000
6,70,166772.416667


In [337]:
df['MSSubClass_Rating'] = 0

mssubclass_sorted = df.groupby('MSSubClass', as_index=False)['SalePrice'].mean().sort_values('SalePrice')

for idx, mssubclass in enumerate(mssubclass_sorted['MSSubClass']):
    df.loc[df['MSSubClass'] == mssubclass, 'MSSubClass_Rating'] = idx + 1

Проверим правильность сортировки:

In [338]:
table_old = df.groupby('MSSubClass', as_index=False)['SalePrice'].mean().sort_values('SalePrice').reset_index(drop=True)
table_new = df.groupby('MSSubClass_Rating', as_index=False)['SalePrice'].mean().sort_values('SalePrice').reset_index(drop=True)
pd.concat([table_old, table_new], axis=1)

,MSSubClass,SalePrice,MSSubClass_Rating,SalePrice
0,30,95829.724638,1,95829.724638
1,180,102300.000000,2,102300.000000
2,45,108591.666667,3,108591.666667
3,190,129613.333333,4,129613.333333
4,90,133541.076923,5,133541.076923
5,160,138647.380952,6,138647.380952
6,50,143302.972222,7,143302.972222
7,85,147810.000000,8,147810.000000
8,40,156125.000000,9,156125.000000
9,70,166772.416667,10,166772.416667


И сразу избавлю от ненужной фичи *MSSubClass*, чтобы потом про неё не забыть (дальшее это буду делать в конце каждого пункта):

In [339]:
df = df.drop(columns='MSSubClass')

## MSZoning fe
Показывает градостроительное назначение земли. В данном признаки 8 значений, но в обучаемых данных используется лишь 5. Если модель увидит какое-то новое значение, предсказание
от этого лучше не станет. Поэтому всем значениям, которые модель не видела, буду присваивать самое популярное значение в тренировочном датасете. Самое популярное значение — *RL*.

Порядок (из EDA): *C (all)* < *RH* < *RM* < *RL* < *FV*

In [340]:
df['MSZoning_Rating'] = 4 # Сразу присваиваю значени RL

df.loc[df['MSZoning'] == 'C (all)', 'MSZoning_Rating'] = 1
df.loc[df['MSZoning'] == 'RH', 'MSZoning_Rating'] = 2
df.loc[df['MSZoning'] == 'RM', 'MSZoning_Rating'] = 3
df.loc[df['MSZoning'] == 'FV', 'MSZoning_Rating'] = 5

df = df.drop(columns='MSZoning')

## LotFrontage fe
Показывает ширину участка, который выходит прям на улицу/дорогу